# 02 GPU training on Colab (T4)

Runs the TerraLens GPU experiments on a single Colab T4. Before running:

1. Locally: `python scripts/make_data_zip.py` and upload `outputs/terralens_data.zip` to Google Drive (default path below: `MyDrive/terralens/terralens_data.zip`).
2. Runtime → Change runtime type → T4 GPU.

The notebook clones the repo, installs `requirements-gpu.txt`, unzips the data, runs the configs in `CONFIGS` one after another and zips `results/` back to Drive. Model selection uses validation only.

In [ ]:
DRIVE_ZIP = '/content/drive/MyDrive/terralens/terralens_data.zip'
REPO = 'https://github.com/shaauryaa/terralens.git'
BRANCH = 'main'
CONFIGS = ['configs/e2_xlmr.yaml', 'configs/e2_muril.yaml', 'configs/e2_indicbertv2.yaml', 'configs/e2_mmbert.yaml']
EXTRA_SEEDS = []          # e.g. [43, 44] for the best config
OUT_ZIP = '/content/drive/MyDrive/terralens/results_colab.zip'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf terralens && git clone -q --branch $BRANCH $REPO terralens
%cd /content/terralens
!pip install -q -r requirements-gpu.txt && pip install -q -e .
!unzip -o -q $DRIVE_ZIP -d /content/terralens && ls data/processed data/splits

In [ ]:
import subprocess, sys, time
def run(cfg, seed=None):
    cmd = [sys.executable, '-m', 'terralens.models.encoder', '--config', cfg]
    if seed is not None:
        cmd += ['--seed', str(seed)]
    t0 = time.time(); r = subprocess.run(cmd)
    print(f'{cfg} seed={seed}: exit {r.returncode} in {(time.time()-t0)/60:.1f} min')
for cfg in CONFIGS:
    run(cfg)
    for s in EXTRA_SEEDS:
        run(cfg, s)

In [ ]:
!python scripts/make_results_table.py || true
!cd /content/terralens && zip -q -r $OUT_ZIP results reports/results.md && ls -la $OUT_ZIP

Download `results_colab.zip` from Drive and unzip it into the repo root; the JSON files go to `results/`.